## Initial Setup

In [ ]:
import os # Configure which GPU
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Import NumPy
import numpy as np
theta = np.linspace(0, np.pi, 100)  # Example: 100 values from 0 to pi
phi = np.linspace(0, 2*np.pi, 100)

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

# Colab does currently not support the latest version of ipython.
# Thus, the preview does not work in Colab. However, whenever possible we
# strongly recommend to use the scene preview mode.
try: # detect if the notebook runs in Colab
    import google.colab
    no_preview = True # deactivate preview
except:
    if os.getenv("SIONNA_NO_PREVIEW"):
        no_preview = True
    else:
        no_preview = False

resolution = [480,320] # increase for higher quality of renderings

# Define magic cell command to skip a cell if needed
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

# Set random seed for reproducibility
sionna.config.seed = 42

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera, Antenna

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement

In [ ]:
# Load integrated scene
scene = load_scene(sionna.rt.scene.munich) # Try also sionna.rt.scene.etoile

In [ ]:
# Render scene
if no_preview:
    scene.render(camera="scene-cam-0", num_samples=512);

In [ ]:
%%skip_if no_preview
# Open 3D preview (only works in Jupyter notebook)
scene.preview()

In [ ]:
%%skip_if no_preview
# Open 3D preview (only works in Jupyter notebook)
scene.preview()

## Antenna Experimentation

In [ ]:
sionna.rt.Antenna(pattern="dipole", polarization="H", polarization_model=2, dtype=tf.complex64)

In [ ]:
Antenna("tr38901", "VH")

In [ ]:
from sionna.rt.antenna import compute_gain, tr38901_pattern, dipole_pattern


In [ ]:
sionna.rt.antenna.compute_gain(tr38901_pattern)

In [ ]:
sionna.rt.antenna.visualize(tr38901_pattern)

In [ ]:
#sionna.rt.antenna.dipole_pattern(theta, phi, slant_angle=9.0, polarization_model=1, dtype=tf.complex64)

In [ ]:
sionna.rt.antenna.visualize(dipole_pattern)

In [ ]:
sionna.rt.antenna.dipole_pattern(theta, phi, slant_angle=0.0, polarization_model=2, dtype=tf.complex64)

In [ ]:
#sionna.rt.antenna.visualize(dipole_pattern)

In [ ]:
sionna.rt.antenna.hw_dipole_pattern(theta, phi, slant_angle=0.0, polarization_model=2, dtype=tf.complex64)

In [ ]:
sionna.rt.antenna.tr38901_pattern(theta, phi, slant_angle=0.0, polarization_model=2, dtype=tf.complex64)

## Implementing MSI to TensorFlow Conversion

In [ ]:
import numpy as np
import tensorflow as tf


In [ ]:
def parse_msi_file(file_path):
    """
    Parses a .msi file with the structure:
        NAME ...
        FREQUENCY ...
        GAIN ...
        TILT ...
        COMMENT ...
        HORIZONTAL 360
           <360 lines of angle, gain>
        VERTICAL 360
           <360 lines of angle, gain>

    Returns:
        metadata (dict): Dictionary containing metadata (NAME, FREQUENCY, GAIN, TILT, COMMENT)
        horizontal_data (numpy array): shape (360, 2) -> [angle, gain_value]
        vertical_data (numpy array): shape (360, 2) -> [angle, gain_value]
    """
    metadata = {
        "NAME": None,
        "FREQUENCY": None,
        "GAIN": None,
        "TILT": None,
        "COMMENT": None
    }

    horizontal_data = []
    vertical_data = []

    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Remove empty lines and strip whitespace
    lines = [line.strip() for line in lines if line.strip()]

    idx = 0
    while idx < len(lines):
        line = lines[idx]

        # Parse known metadata lines
        if line.startswith("NAME"):
            metadata["NAME"] = line.split(" ", 1)[1]  # everything after "NAME "
        elif line.startswith("FREQUENCY"):
            parts = line.split()
            if len(parts) > 1:
                metadata["FREQUENCY"] = parts[1]
        elif line.startswith("GAIN"):
            # e.g. "GAIN 11.33 dBd"
            parts = line.split()
            if len(parts) >= 2:
                metadata["GAIN"] = parts[1]  # "11.33"
        elif line.startswith("TILT"):
            metadata["TILT"] = line.split(" ", 1)[1]
        elif line.startswith("COMMENT"):
            metadata["COMMENT"] = line.split(" ", 1)[1]

        # If we hit the "HORIZONTAL 360" marker
        elif line.upper().startswith("HORIZONTAL 360"):
            idx += 1
            for _ in range(360):
                angle_line = lines[idx]
                angle_str, val_str = angle_line.split()
                horizontal_data.append([float(angle_str), float(val_str)])
                idx += 1
            continue  # skip the idx += 1 at the bottom of the loop

        # If we hit the "VERTICAL 360" marker
        elif line.upper().startswith("VERTICAL 360"):
            idx += 1
            for _ in range(360):
                angle_line = lines[idx]
                angle_str, val_str = angle_line.split()
                vertical_data.append([float(angle_str), float(val_str)])
                idx += 1
            continue

        idx += 1

    # Convert lists to numpy arrays
    horizontal_data = np.array(horizontal_data)
    vertical_data = np.array(vertical_data)

    return metadata, horizontal_data, vertical_data


In [ ]:
import numpy as np

def parse_motorola_msi(file_path):
    """
    Parses a Motorola-style .msi file with the following structure:

        CanopyIntegrAntennas    <-- (Line 1, treated as 'NAME')
        FREQUENCY 5300
        GAIN (dBi) 10
        TILT 0
        COMMENT (possibly empty)
        HORIZONTAL 360
           <360 lines of angle, gain_value>
        VERTICAL 360
           <360 lines of angle, gain_value>

    Returns:
        metadata (dict): Contains fields like NAME, FREQUENCY, GAIN, TILT, COMMENT
        horizontal_data (numpy.ndarray): shape (360, 2) -> [angle, gain_value]
        vertical_data (numpy.ndarray): shape (360, 2) -> [angle, gain_value]
    """

    metadata = {
        "NAME": None,
        "FREQUENCY": None,
        "GAIN": None,   # Numeric gain (dBi)
        "TILT": None,
        "COMMENT": None
    }

    horizontal_data = []
    vertical_data = []

    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Clean up lines (strip and remove empty ones)
    lines = [line.strip() for line in lines if line.strip()]

    idx = 0
    while idx < len(lines):
        line = lines[idx]

        # 1) First line is usually the antenna name if not explicitly labeled
        if idx == 0 and not line.startswith(("NAME", "FREQUENCY")):
            metadata["NAME"] = line
            idx += 1
            continue

        # 2) Extract Frequency
        if line.startswith("FREQUENCY"):
            parts = line.split()
            metadata["FREQUENCY"] = float(parts[1]) if len(parts) > 1 else None

        # 3) Extract Gain (dBi)
        elif line.startswith("GAIN"):
            parts = line.split()
            if len(parts) >= 3:
                metadata["GAIN"] = float(parts[-1])  # Last value is assumed to be the numeric gain
            else:
                metadata["GAIN"] = float(parts[1]) if len(parts) > 1 else None

        # 4) Extract Tilt
        elif line.startswith("TILT"):
            parts = line.split()
            metadata["TILT"] = float(parts[1]) if len(parts) > 1 else None

        # 5) Extract Comment (can be empty)
        elif line.startswith("COMMENT"):
            parts = line.split(" ", 1)
            metadata["COMMENT"] = parts[1] if len(parts) > 1 else ""

        # 6) Extract Horizontal Pattern Data
        elif line.upper().startswith("HORIZONTAL 360"):
            idx += 1
            for _ in range(360):
                angle, gain = lines[idx].split()
                horizontal_data.append([float(angle), float(gain)])
                idx += 1
            continue

        # 7) Extract Vertical Pattern Data
        elif line.upper().startswith("VERTICAL 360"):
            idx += 1
            for _ in range(360):
                angle, gain = lines[idx].split()
                vertical_data.append([float(angle), float(gain)])
                idx += 1
            continue

        idx += 1

    # Convert lists to NumPy arrays
    horizontal_data = np.array(horizontal_data)
    vertical_data = np.array(vertical_data)
    
    return metadata, horizontal_data, vertical_data

In [ ]:
def normalize_pattern(data):
    """
    Normalizes the pattern gain to [0, 1].
    data shape is (N, 2): [angle, gain_value]

    Returns the same shape (N, 2) with normalized gains.
    """
    angles = data[:, 0]
    gains = data[:, 1]
    min_gain, max_gain = np.min(gains), np.max(gains)

    if max_gain == min_gain:
        norm_gains = gains - min_gain  # all zeros
    else:
        norm_gains = (gains - min_gain) / (max_gain - min_gain)

    return np.column_stack((angles, norm_gains))


In [ ]:
def create_tf_dataset(horizontal_data, vertical_data):
    """
    Convert horizontal and vertical pattern data into a single tf.data.Dataset.
    Each element is a dictionary with:
      {
        "horizontal_angle": ...,
        "horizontal_gain": ...,
        "vertical_angle": ...,
        "vertical_gain": ...
      }
    """
    ds_dicts = []
    for i in range(len(horizontal_data)):
        ds_dicts.append({
            "horizontal_angle": horizontal_data[i, 0],
            "horizontal_gain": horizontal_data[i, 1],
            "vertical_angle": vertical_data[i, 0],
            "vertical_gain": vertical_data[i, 1]
        })

    dataset = tf.data.Dataset.from_generator(
        lambda: ds_dicts,
        output_signature={
            "horizontal_angle": tf.TensorSpec(shape=(), dtype=tf.float32),
            "horizontal_gain": tf.TensorSpec(shape=(), dtype=tf.float32),
            "vertical_angle": tf.TensorSpec(shape=(), dtype=tf.float32),
            "vertical_gain": tf.TensorSpec(shape=(), dtype=tf.float32),
        }
    )
    return dataset


#### Huawei

In [ ]:
# Specify MSI file path
msi_file_path = "AHP4518R3v06_0699_X_CO_M45_00T_Lr1.msi"

# 1. Parse the file
metadata, horizontal_data, vertical_data = parse_msi_file(msi_file_path)
print("Metadata:", metadata)
print("Horizontal data (first 5 rows):\n", horizontal_data[:5])
print("Vertical data (first 5 rows):\n", vertical_data[:5])

# 2. Normalize patterns
horizontal_norm = normalize_pattern(horizontal_data)
vertical_norm = normalize_pattern(vertical_data)

# 3. Create TensorFlow dataset
huawei_tf_dataset = create_tf_dataset(horizontal_norm, vertical_norm)

# 4. Preview the dataset
for sample in huawei_tf_dataset.take(5):
    print(sample)


In [ ]:
huawei_tf_dataset

In [ ]:
import matplotlib.pyplot as plt

# Plot horizontal pattern
plt.figure(figsize=(8, 4))
plt.plot(horizontal_data[:, 0], horizontal_data[:, 1], label='Horizontal Raw')
plt.plot(horizontal_norm[:, 0], horizontal_norm[:, 1], label='Horizontal Normalized')
plt.legend()
plt.xlabel('Angle (degrees)')
plt.ylabel('Gain')
plt.title('Horizontal Pattern')
plt.grid(True)
plt.show()

# Plot vertical pattern
plt.figure(figsize=(8, 4))
plt.plot(vertical_data[:, 0], vertical_data[:, 1], label='Vertical Raw')
plt.plot(vertical_norm[:, 0], vertical_norm[:, 1], label='Vertical Normalized')
plt.legend()
plt.xlabel('Angle (degrees)')
plt.ylabel('Gain')
plt.title('Vertical Pattern')
plt.grid(True)
plt.show()


In [ ]:
motorola_tf_dataset

### today

In [ ]:
import tensorflow as tf
import numpy as np

# Load MSI file and preprocess
#msi_file_path = "AHP4518R3v06_0699_X_CO_M45_00T_Lr1.msi"
#metadata, horizontal_data, vertical_data = parse_msi_file(msi_file_path)
#msi_file_path = "Canopy.msi"
msi_file_path = "synthetic_antenna.msi"
metadata, horizontal_data, vertical_data = parse_motorola_msi(msi_file_path)

# Normalize gain values
horizontal_norm = normalize_pattern(horizontal_data)
vertical_norm = normalize_pattern(vertical_data)

# Convert to TensorFlow tensors for fast lookup
horizontal_angles_tf = tf.constant(horizontal_norm[:, 0], dtype=tf.float32)  # Angles
horizontal_gains_tf = tf.constant(horizontal_norm[:, 1], dtype=tf.float32)  # Gains

vertical_angles_tf = tf.constant(vertical_norm[:, 0], dtype=tf.float32)  # Angles
vertical_gains_tf = tf.constant(vertical_norm[:, 1], dtype=tf.float32)  # Gains

'''
def antenna_pattern(theta, phi):
    """
    Retrieves or interpolates the precomputed antenna gain pattern.

    Inputs:
    - theta: array-like (zenith angle in radians) of shape (N,)
    - phi: array-like (azimuth angle in radians) of shape (N,)

    Outputs:
    - horizontal_gain: Tensor (gain for horizontal pattern) of shape (N,)
    - vertical_gain: Tensor (gain for vertical pattern) of shape (N,)
    """

    # Ensure input tensors are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Flatten theta and phi to ensure they are 1D arrays
    original_shape = theta.shape  # Store original shape
    theta = tf.reshape(theta, [-1])  # Flatten to (N,)
    phi = tf.reshape(phi, [-1])  # Flatten to (N,)

    # Convert angles from radians to degrees for lookup
    theta_deg = theta * (180.0 / np.pi)  # Convert to degrees
    phi_deg = phi * (180.0 / np.pi)  # Convert to degrees

    # Wrap angles to [0, 360) to match dataset
    theta_deg = tf.math.mod(theta_deg, 360.0)
    phi_deg = tf.math.mod(phi_deg, 360.0)

    # Compute absolute differences for all pairs
    horizontal_diff = tf.abs(horizontal_angles_tf - tf.reshape(phi_deg, [-1, 1]))  # (N, 360)
    vertical_diff = tf.abs(vertical_angles_tf - tf.reshape(theta_deg, [-1, 1]))  # (N, 360)

    # Find the closest matching indices
    horizontal_idx = tf.argmin(horizontal_diff, axis=1)
    vertical_idx = tf.argmin(vertical_diff, axis=1)

    # Retrieve corresponding gain values
    horizontal_gain = tf.gather(horizontal_gains_tf, horizontal_idx)
    vertical_gain = tf.gather(vertical_gains_tf, vertical_idx)

    # Reshape to original input shape (e.g., (50,50) for visualization)
    horizontal_gain = tf.reshape(horizontal_gain, original_shape)
    vertical_gain = tf.reshape(vertical_gain, original_shape)

    

    return tf.cast(horizontal_gain, dtype= tf.complex64), tf.cast(vertical_gain, dtype=tf.complex64)

'''

'''
def antenna_pattern(theta, phi):
    """
    Retrieves or interpolates the precomputed antenna gain pattern.

    Inputs:
      - theta: float or array-like (zenith angle in radians)
      - phi: float or array-like (azimuth angle in radians)

    Outputs:
      - horizontal_gain: Tensor (gain for horizontal pattern)
      - vertical_gain: Tensor (gain for vertical pattern)
    """

    # Ensure input angles are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Convert angles from radians to degrees
    theta_deg = theta * (180.0 / np.pi)
    phi_deg = phi * (180.0 / np.pi)

    # Wrap phi to [0, 360)
    phi_deg = tf.math.mod(phi_deg, 360.0)

    # Adjust theta to full 360° mapping for the vertical pattern
    vertical_lookup_angle = tf.math.mod(2 * theta_deg, 360.0)

    # Expand for broadcasting in TensorFlow
    phi_exp = tf.reshape(phi_deg, [-1, 1])  # Shape (N, 1)
    theta_exp = tf.reshape(vertical_lookup_angle, [-1, 1])  # Shape (N, 1)

    # Compute absolute differences for lookup
    horizontal_diff = tf.abs(horizontal_angles_tf - phi_exp)  # (N, 360)
    vertical_diff = tf.abs(vertical_angles_tf - theta_exp)  # (N, 360)

    # Find the two closest indices for averaging
    horizontal_sorted_idx = tf.argsort(horizontal_diff, axis=1)
    horizontal_two_idx = horizontal_sorted_idx[:, :2]

    vertical_sorted_idx = tf.argsort(vertical_diff, axis=1)
    vertical_two_idx = vertical_sorted_idx[:, :2]

    # Gather and average the two nearest gain values
    horizontal_gains_nearest = tf.gather(horizontal_gains_tf, horizontal_two_idx)
    horizontal_gain_avg = tf.reduce_mean(horizontal_gains_nearest, axis=1)

    vertical_gains_nearest = tf.gather(vertical_gains_tf, vertical_two_idx)
    vertical_gain_avg = tf.reduce_mean(vertical_gains_nearest, axis=1)

    # Reshape outputs to match input shape (ensures compatibility for single values and arrays)
    horizontal_gain_avg = tf.reshape(horizontal_gain_avg, theta.shape)
    vertical_gain_avg = tf.reshape(vertical_gain_avg, theta.shape)

    # Cast output to `tf.complex64` (Fix for visualization)
    return tf.cast(horizontal_gain_avg, tf.complex64), tf.cast(vertical_gain_avg, tf.complex64)
'''
def antenna_pattern(theta, phi):
    """
    Retrieves or interpolates the precomputed antenna gain pattern.

    Inputs:
      - theta: float or array-like (zenith angle in radians)
      - phi: float or array-like (azimuth angle in radians)

    Outputs:
      - horizontal_gain: Tensor (gain for horizontal pattern)
      - vertical_gain: Tensor (gain for vertical pattern)
    """
    print(theta)
    # Ensure input angles are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Convert angles from radians to degrees
    theta_deg = theta * (180.0 / np.pi)
    phi_deg = phi * (180.0 / np.pi)

    # Wrap phi to [0, 360)
    phi_deg = tf.math.mod(phi_deg, 360.0)

    # Adjust theta to full 360° mapping for the vertical pattern
    vertical_lookup_angle = tf.math.mod(2 * theta_deg, 360.0)

    # Expand for broadcasting in TensorFlow
    phi_exp = tf.reshape(phi_deg, [-1, 1])  # Shape (N, 1)
    theta_exp = tf.reshape(vertical_lookup_angle, [-1, 1])  # Shape (N, 1)

    # Compute absolute differences for lookup
    horizontal_diff = tf.abs(horizontal_angles_tf - phi_exp)  # (N, 360)
    vertical_diff = tf.abs(vertical_angles_tf - theta_exp)  # (N, 360)

    # Find the two closest indices for averaging
    horizontal_sorted_idx = tf.argsort(horizontal_diff, axis=1)
    horizontal_two_idx = horizontal_sorted_idx[:, :2]

    vertical_sorted_idx = tf.argsort(vertical_diff, axis=1)
    vertical_two_idx = vertical_sorted_idx[:, :2]

    # Gather and average the two nearest gain values
    horizontal_gains_nearest = tf.gather(horizontal_gains_tf, horizontal_two_idx)
    horizontal_gain_avg = tf.reduce_mean(horizontal_gains_nearest, axis=1)

    vertical_gains_nearest = tf.gather(vertical_gains_tf, vertical_two_idx)
    vertical_gain_avg = tf.reduce_mean(vertical_gains_nearest, axis=1)

    # Reshape outputs to match input shape (ensures compatibility for single values and arrays)
    horizontal_gain_avg = tf.reshape(horizontal_gain_avg, theta.shape)
    vertical_gain_avg = tf.reshape(vertical_gain_avg, theta.shape)

    # Cast output to `tf.complex64` (Fix for visualization)
    print(horizontal_gain_avg)
    return tf.cast(horizontal_gain_avg, tf.complex64), tf.cast(vertical_gain_avg, tf.complex64)



'''
def antenna_pattern(theta, phi):
    """
    Retrieves or interpolates the precomputed antenna gain pattern.

    Inputs:
      - theta: float or array-like (zenith angle in radians)
      - phi: float or array-like (azimuth angle in radians)

    Outputs:
      - horizontal_gain: Tensor (gain for horizontal pattern)
      - vertical_gain: Tensor (gain for vertical pattern)
    """

    # Ensure input angles are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Convert angles from radians to degrees
    theta_deg = theta * (180.0 / np.pi)
    phi_deg = phi * (180.0 / np.pi)

    # Wrap angles within [0, 360) for consistency
    phi_deg = tf.math.mod(phi_deg, 360.0)
    theta_deg = tf.math.mod(theta_deg, 360.0)

    #  **Fix: Ensure phi_deg and theta_deg are at least 1D tensors**
    phi_deg = tf.reshape(phi_deg, [-1])  # Ensures shape (N,)
    theta_deg = tf.reshape(theta_deg, [-1])  # Ensures shape (N,)

    #  **Fix: Ensure horizontal_angles_tf is sorted**
    horizontal_angles_tf_sorted = tf.sort(horizontal_angles_tf)
    vertical_angles_tf_sorted = tf.sort(vertical_angles_tf)

    ### ** Interpolation Using `tf.searchsorted()`**

    ## **Horizontal Pattern Interpolation**
    h_idx = tf.searchsorted(horizontal_angles_tf_sorted, phi_deg, side='left')

    # Prevent index out-of-bounds
    h_idx1 = tf.clip_by_value(h_idx - 1, 0, horizontal_angles_tf_sorted.shape[0] - 1)
    h_idx2 = tf.clip_by_value(h_idx, 0, horizontal_angles_tf_sorted.shape[0] - 1)

    # Retrieve angle and gain values for interpolation
    h_angle1, h_angle2 = tf.gather(horizontal_angles_tf_sorted, h_idx1), tf.gather(horizontal_angles_tf_sorted, h_idx2)
    h_gain1, h_gain2 = tf.gather(horizontal_gains_tf, h_idx1), tf.gather(horizontal_gains_tf, h_idx2)

    # Compute interpolation weights
    h_weight = (phi_deg - h_angle1) / (h_angle2 - h_angle1 + 1e-9)

    # Linearly interpolate gains
    horizontal_gain = (1 - h_weight) * h_gain1 + h_weight * h_gain2

    ## **Vertical Pattern Interpolation**
    v_idx = tf.searchsorted(vertical_angles_tf_sorted, theta_deg, side='left')

    # Prevent index out-of-bounds
    v_idx1 = tf.clip_by_value(v_idx - 1, 0, vertical_angles_tf_sorted.shape[0] - 1)
    v_idx2 = tf.clip_by_value(v_idx, 0, vertical_angles_tf_sorted.shape[0] - 1)

    # Retrieve angle and gain values for interpolation
    v_angle1, v_angle2 = tf.gather(vertical_angles_tf_sorted, v_idx1), tf.gather(vertical_angles_tf_sorted, v_idx2)
    v_gain1, v_gain2 = tf.gather(vertical_gains_tf, v_idx1), tf.gather(vertical_gains_tf, v_idx2)

    # Compute interpolation weights
    v_weight = (theta_deg - v_angle1) / (v_angle2 - v_angle1 + 1e-9)

    # Linearly interpolate gains
    vertical_gain = (1 - v_weight) * v_gain1 + v_weight * v_gain2

    ### ** Ensure Output Shape Matches Input**
    horizontal_gain = tf.reshape(horizontal_gain, theta.shape)
    vertical_gain = tf.reshape(vertical_gain, theta.shape)

    # Cast output to `tf.complex64` for visualization compatibility
    return tf.cast(horizontal_gain, tf.complex64), tf.cast(vertical_gain, tf.complex64)

'''






# Test the function
test_theta = 45.0  # Example zenith angle
test_phi = 90.0    # Example azimuth angle
h_gain, v_gain = antenna_pattern(test_theta, test_phi)

# Print results
print(f"Horizontal Gain at (theta={test_theta}, phi={test_phi}):", h_gain.numpy())
print(f"Vertical Gain at (theta={test_theta}, phi={test_phi}):", v_gain.numpy())


### Motorola

In [ ]:
def parse_msi_file(file_path):
    """
    Parses a .msi-like file with the following structure:

        CanopyIntegrAntennas    <-- (Line 1, treat as 'NAME')
        FREQUENCY 5300
        GAIN (dBi) 10
        TILT 0
        COMMENT (possibly empty)
        HORIZONTAL 360
           <360 lines of angle, gain_value>
        VERTICAL 360
           <360 lines of angle, gain_value>

    Returns:
        metadata (dict): Contains fields like NAME, FREQUENCY, GAIN, TILT, COMMENT
        horizontal_data (numpy.ndarray): shape (360, 2) -> [angle, gain_value]
        vertical_data (numpy.ndarray): shape (360, 2) -> [angle, gain_value]
    """

    # Initialize default metadata
    metadata = {
        "NAME": None,
        "FREQUENCY": None,
        "GAIN": None,   # Will store numeric gain
        "TILT": None,
        "COMMENT": None
    }

    horizontal_data = []
    vertical_data = []

    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Clean up lines (strip and remove empty ones)
    lines = [line.strip() for line in lines if line.strip()]

    idx = 0
    while idx < len(lines):
        line = lines[idx]

        # 1) The first line might be "CanopyIntegrAntennas" (treat as 'NAME' if no "NAME" keyword is present)
        if idx == 0 and not line.startswith("NAME") and not line.startswith("FREQUENCY"):
            metadata["NAME"] = line
            idx += 1
            continue

        # 2) FREQUENCY line
        if line.startswith("FREQUENCY"):
            # e.g. "FREQUENCY 5300"
            parts = line.split()
            if len(parts) > 1:
                metadata["FREQUENCY"] = parts[1]  # "5300"
        
        # 3) GAIN line
        elif line.startswith("GAIN"):
            # e.g. "GAIN (dBi) 10"
            # We'll grab the last token as the numeric gain
            parts = line.split()
            # Usually: ["GAIN", "(dBi)", "10"] or something similar
            if len(parts) >= 3:
                # The last part should be the numeric gain
                metadata["GAIN"] = parts[-1]  # "10"
            else:
                # fallback if format is different
                metadata["GAIN"] = parts[1] if len(parts) > 1 else None
        
        # 4) TILT line
        elif line.startswith("TILT"):
            # e.g. "TILT 0"
            parts = line.split()
            if len(parts) > 1:
                metadata["TILT"] = parts[1]
        
        # 5) COMMENT line (could be empty after "COMMENT")
        elif line.startswith("COMMENT"):
            # We split once to keep the possible remainder
            parts = line.split(" ", 1)
            if len(parts) > 1:
                metadata["COMMENT"] = parts[1]
            else:
                metadata["COMMENT"] = ""  # empty if no comment after the word "COMMENT"
        
        # 6) HORIZONTAL 360
        elif line.upper().startswith("HORIZONTAL 360"):
            idx += 1
            for _ in range(360):
                angle_line = lines[idx]
                angle_str, val_str = angle_line.split()
                horizontal_data.append([float(angle_str), float(val_str)])
                idx += 1
            continue  # skip the idx increment at the end of loop

        # 7) VERTICAL 360
        elif line.upper().startswith("VERTICAL 360"):
            idx += 1
            for _ in range(360):
                angle_line = lines[idx]
                angle_str, val_str = angle_line.split()
                vertical_data.append([float(angle_str), float(val_str)])
                idx += 1
            continue

        idx += 1

    horizontal_data = np.array(horizontal_data)
    vertical_data = np.array(vertical_data)
    
    return metadata, horizontal_data, vertical_data


In [ ]:
def normalize_pattern(data):
    """
    Normalizes the pattern gain to [0, 1].
    data shape is (N, 2): [angle, gain_value]

    Returns the same shape (N, 2) with normalized gains.
    """
    angles = data[:, 0]
    gains = data[:, 1]
    min_gain, max_gain = np.min(gains), np.max(gains)

    if min_gain == max_gain:
        # If all gains are the same, normalization would be zero
        norm_gains = gains - min_gain
    else:
        norm_gains = (gains - min_gain) / (max_gain - min_gain)

    return np.column_stack((angles, norm_gains))


In [ ]:
def create_tf_dataset(horizontal_data, vertical_data):
    """
    Converts horizontal and vertical pattern data into a single tf.data.Dataset.
    Each element is a dictionary with keys:
        "horizontal_angle", "horizontal_gain", "vertical_angle", "vertical_gain".
    """
    data_dicts = []
    for i in range(len(horizontal_data)):
        data_dicts.append({
            "horizontal_angle": horizontal_data[i, 0],
            "horizontal_gain": horizontal_data[i, 1],
            "vertical_angle": vertical_data[i, 0],
            "vertical_gain": vertical_data[i, 1]
        })

    ds = tf.data.Dataset.from_generator(
        lambda: data_dicts,
        output_signature={
            "horizontal_angle": tf.TensorSpec(shape=(), dtype=tf.float32),
            "horizontal_gain": tf.TensorSpec(shape=(), dtype=tf.float32),
            "vertical_angle": tf.TensorSpec(shape=(), dtype=tf.float32),
            "vertical_gain": tf.TensorSpec(shape=(), dtype=tf.float32),
        }
    )
    return ds


In [ ]:
# Set your .msi-like file path
#msi_file_path = "Canopy.msi"  # Change if needed
msi_file_path = "synthetic_antenna.msi"


# 1. Parse the file
metadata, horizontal_data, vertical_data = parse_msi_file(msi_file_path)
print("Metadata:\n", metadata)
print("\nHorizontal data (first 5 rows):")
print(horizontal_data[:5])
print("\nVertical data (first 5 rows):")
print(vertical_data[:5])

# 2. Normalize patterns
horizontal_norm = normalize_pattern(horizontal_data)
vertical_norm = normalize_pattern(vertical_data)

print("\nNormalized Horizontal data (first 5 rows):")
print(horizontal_norm[:5])
print("\nNormalized Vertical data (first 5 rows):")
print(vertical_norm[:5])

# 3. Create TensorFlow dataset
motorola_tf_dataset = create_tf_dataset(horizontal_norm, vertical_norm)

# 4. Preview the dataset
print("\nPreviewing the tf.data.Dataset:")
for sample in tf_dataset.take(5):
    print(sample)


In [ ]:
import matplotlib.pyplot as plt

# Plot horizontal pattern (raw vs normalized)
plt.figure(figsize=(10, 4))
plt.plot(horizontal_data[:, 0], horizontal_data[:, 1], label='Horizontal Raw')
plt.plot(horizontal_norm[:, 0], horizontal_norm[:, 1], label='Horizontal Normalized')
plt.xlabel('Angle (degrees)')
plt.ylabel('Gain')
plt.title('Horizontal Pattern')
plt.legend()
plt.grid(True)
plt.show()

# Plot vertical pattern (raw vs normalized)
plt.figure(figsize=(10, 4))
plt.plot(vertical_data[:, 0], vertical_data[:, 1], label='Vertical Raw')
plt.plot(vertical_norm[:, 0], vertical_norm[:, 1], label='Vertical Normalized')
plt.xlabel('Angle (degrees)')
plt.ylabel('Gain')
plt.title('Vertical Pattern')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
motorola_tf_dataset

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Generate 1000 values from 0 to π for theta
test_theta_values = np.linspace(0, np.pi, 1000)
test_phi_values = np.zeros_like(test_theta_values)  # Keep phi constant at 0

# Convert to TensorFlow tensors
test_theta_tf = tf.constant(test_theta_values, dtype=tf.float32)
test_phi_tf = tf.constant(test_phi_values, dtype=tf.float32)

# Pass the test values into the antenna pattern function
h_gain, v_gain = antenna_pattern(test_theta_tf, test_phi_tf)

# Convert results to numpy for visualization
h_gain_np = h_gain.numpy().real  # Extract real part since it's complex
v_gain_np = v_gain.numpy().real

# Plot the results to see how the gain varies with theta
plt.figure(figsize=(10, 5))
plt.plot(test_theta_values * (180.0 / np.pi), h_gain_np, label="Horizontal Gain")
plt.plot(test_theta_values * (180.0 / np.pi), v_gain_np, label="Vertical Gain")
plt.xlabel("Theta (Degrees)")
plt.ylabel("Gain")
plt.title("Gain vs Theta (phi=0 constant)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import tensorflow as tf
import numpy as np
import sionna.rt.antenna  # Import Sionna's antenna module

def test_antenna_pattern_visualization():
    """
    Test function for visualizing the antenna pattern.

    This function generates theta values from 0 to π in 1000 steps, while
    keeping phi constant at 0, and passes them through Sionna's visualize method.
    """

    # Generate 1000 theta values from 0 to π
    test_theta = tf.linspace(0.0, np.pi, 1000)
    test_phi = tf.zeros_like(test_theta)  # Keep phi = 0 for all points

    # Run visualization using Sionna's built-in method
    sionna.rt.antenna.visualize(lambda theta, phi: antenna_pattern(theta, phi))

# Call the test function to visualize the pattern
test_antenna_pattern_visualization()


In [ ]:
import tensorflow as tf
import numpy as np
import sionna.rt.antenna

def test_antenna_pattern(theta, phi):
    """
    A test antenna pattern function that smoothly varies with theta and phi.

    Inputs:
      - theta: Tensor (zenith angle in radians), expected shape (N, M)
      - phi: Tensor (azimuth angle in radians), expected shape (N, M)

    Outputs:
      - horizontal_gain: Complex tensor with horizontal pattern gain
      - vertical_gain: Complex tensor with vertical pattern gain
    """

    # Ensure inputs are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Compute real and imaginary parts separately
    exp_phi = tf.complex(tf.cos(phi), tf.sin(phi))  # Equivalent to exp(1j * phi)
    
    # Example test pattern: Using simple sinusoidal variations
    horizontal_gain = tf.complex(tf.cos(theta), 0.0) * exp_phi  # Cosine for smooth variation
    vertical_gain = tf.complex(tf.sin(theta), 0.0) * tf.math.conj(exp_phi)  # Sine for smooth variation

    return tf.cast(horizontal_gain, tf.complex64), tf.cast(vertical_gain, tf.complex64)

# Function to correctly visualize the pattern
def test_visualize_antenna_pattern():
    """
    Uses Sionna's visualize method to test the custom test_antenna_pattern function.
    """

    # Pass `theta` and `phi` directly from Sionna to the pattern function
    sionna.rt.antenna.visualize(test_antenna_pattern)

# Call the test function
test_visualize_antenna_pattern()


In [ ]:
import tensorflow as tf
import numpy as np
import sionna.rt.antenna

def test_antenna_pattern(theta, phi):
    """
    A test antenna pattern function that smoothly varies with theta and phi.

    Inputs:
      - theta: Tensor (zenith angle in radians), expected shape (N, M)
      - phi: Tensor (azimuth angle in radians), expected shape (N, M)

    Outputs:
      - horizontal_gain: Complex tensor with horizontal pattern gain
      - vertical_gain: Complex tensor with vertical pattern gain
    """

    # Ensure inputs are float32
    theta = tf.cast(theta, tf.float32)
    phi = tf.cast(phi, tf.float32)

    # Compute real and imaginary parts separately
    exp_phi = tf.complex(tf.cos(phi), tf.sin(phi))  # Equivalent to exp(1j * phi)
    
    # Example test pattern: Using simple sinusoidal variations
    horizontal_gain = tf.complex(tf.cos(theta), 0.0) * exp_phi  # Cosine for smooth variation
    vertical_gain = tf.complex(tf.sin(theta), 0.0) * tf.math.conj(exp_phi)  # Sine for smooth variation

    return tf.cast(horizontal_gain, tf.complex64), tf.cast(vertical_gain, tf.complex64)

# Function to ensure visualization is done with 1000 theta values
def test_visualize_antenna_pattern():
    """
    Uses Sionna's visualize method to test the custom test_antenna_pattern function
    with 1000 theta values spanning from 0 to 2π while keeping correct (50,50) shape.
    """

    # Generate 1000 values for theta (spanning 0 to 2π)
    theta_values = np.linspace(0.0, 2.0 * np.pi, 1000)
    phi_values = np.linspace(-np.pi, np.pi, 50)  # Azimuth values

    # Create a proper meshgrid of shape (50, 50)
    theta_grid, phi_grid = np.meshgrid(theta_values[:50], phi_values, indexing="ij")  # Ensures shape (50, 50)

    # Convert to TensorFlow tensors
    theta_grid_tf = tf.constant(theta_grid, dtype=tf.float32)
    phi_grid_tf = tf.constant(phi_grid, dtype=tf.float32)

    # Pass correctly shaped theta and phi values into visualize()
    sionna.rt.antenna.visualize(lambda theta, phi: test_antenna_pattern(theta_grid_tf, phi_grid_tf))

# Call the test function
test_visualize_antenna_pattern()


## Modifying Sionna antenna patterns

In [ ]:
from sionna.rt.antenna import polarization_model_2
PI = tf.constant(np.pi, dtype=tf.float32)

def tr38901_pattern(theta, phi, slant_angle=0.0,
                    polarization_model=2, dtype=tf.complex64):
    r"""
    Antenna pattern from 3GPP TR 38.901 (Table 7.3-1) [TR38901]_

    Input
    -----
    theta: array_like, float
        Zenith angles wrapped within [0,pi] [rad]

    phi: array_like, float
        Azimuth angles wrapped within [-pi, pi) [rad]

    slant_angle: float
        Slant angle of the linear polarization [rad].
        A slant angle of zero means vertical polarization.

    polarization_model: int, one of [1,2]
        Polarization model to be used. Options `1` and `2`
        refer to :func:`~sionna.rt.antenna.polarization_model_1`
        and :func:`~sionna.rt.antenna.polarization_model_2`,
        respectively.
        Defaults to `2`.

    dtype : tf.complex64 or tf.complex128
        Datatype.
        Defaults to `tf.complex64`.

    Output
    ------
    c_theta: array_like, complex
        Zenith pattern

    c_phi: array_like, complex
        Azimuth pattern


    .. figure:: ../figures/tr38901_pattern.png
        :align: center
    """
    rdtype = dtype.real_dtype
    theta = tf.cast(theta, rdtype)
    phi = tf.cast(phi, rdtype)
    slant_angle = tf.cast(slant_angle, rdtype)

    # Wrap phi to [-PI,PI]
    phi = tf.math.floormod(phi+PI, 2*PI)-PI

    if not theta.shape==phi.shape:
        raise ValueError("theta and phi must have the same shape.")
    if polarization_model not in [1,2]:
        raise ValueError("polarization_model must be 1 or 2")
    theta_3db = phi_3db = tf.cast(65/180*PI, rdtype)
    a_max = sla_v = 30
    g_e_max = 8
    a_v = -tf.minimum(12*((theta-PI/2)/theta_3db)**2, sla_v)
    a_h = -tf.minimum(12*(phi/phi_3db)**2, a_max)
    a_db = -tf.minimum(-(a_v + a_h), a_max) + g_e_max
    a = 10**(a_db/10)
    c = tf.complex(tf.sqrt(a), tf.zeros_like(a))
    if polarization_model==1:
        return polarization_model_1(c, theta, phi, slant_angle)
    else:
        return polarization_model_2(c, slant_angle)



In [ ]:
#test_1 = tr38901_pattern(theta, phi, slant_angle=0.0, polarization_model=2, dtype=tf.complex64)

In [ ]:
test_1

In [ ]:
def tr38901_testing(theta, phi, slant_angle=0.0,
                    polarization_model=2, dtype=tf.complex64):
    r"""
    Antenna pattern from 3GPP TR 38.901 (Table 7.3-1) [TR38901]_

    Input
    -----
    theta: array_like, float
        Zenith angles wrapped within [0,pi] [rad]

    phi: array_like, float
        Azimuth angles wrapped within [-pi, pi) [rad]

    slant_angle: float
        Slant angle of the linear polarization [rad].
        A slant angle of zero means vertical polarization.

    polarization_model: int, one of [1,2]
        Polarization model to be used. Options `1` and `2`
        refer to :func:`~sionna.rt.antenna.polarization_model_1`
        and :func:`~sionna.rt.antenna.polarization_model_2`,
        respectively.
        Defaults to `2`.

    dtype : tf.complex64 or tf.complex128
        Datatype.
        Defaults to `tf.complex64`.

    Output
    ------
    c_theta: array_like, complex
        Zenith pattern

    c_phi: array_like, complex
        Azimuth pattern


    .. figure:: ../figures/tr38901_pattern.png
        :align: center
    """
    rdtype = dtype.real_dtype
    theta = tf.cast(theta, rdtype)
    phi = tf.cast(phi, rdtype)
    slant_angle = tf.cast(slant_angle, rdtype)

    # Wrap phi to [-PI,PI]
    phi = tf.math.floormod(phi+PI, 2*PI)-PI

    if not theta.shape==phi.shape:
        raise ValueError("theta and phi must have the same shape.")
    if polarization_model not in [1,2]:
        raise ValueError("polarization_model must be 1 or 2")
    theta_3db = phi_3db = tf.cast(90/180*PI, rdtype)
    a_max = sla_v = 30
    g_e_max = 8
    a_v = -tf.minimum(12*((theta-PI/2)/theta_3db)**2, sla_v)
    a_h = -tf.minimum(12*(phi/phi_3db)**2, a_max)
    a_db = -tf.minimum(-(a_v + a_h), a_max) + g_e_max
    a = 10**(a_db/10)
    c = tf.complex(tf.sqrt(a), tf.zeros_like(a))
    if polarization_model==1:
        return polarization_model_1(c, theta, phi, slant_angle)
    else:
        return polarization_model_2(c, slant_angle)



In [ ]:
#test_2 = tr38901_pattern(theta, phi, slant_angle=0.0, polarization_model=2, dtype=tf.complex64)

## Continuing tutorial

In [ ]:
#test_2

In [ ]:
# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern=antenna_pattern)#"tr38901",
                             #polarization="V")

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="cross")

# Create transmitter
tx = Transmitter(name="tx",
                 position=[8.5,21,27])

# Add transmitter instance to scene
scene.add(tx)

# Create a receiver
rx = Receiver(name="rx",
              position=[45,90,1.5],
              orientation=[0,0,0])

# Add receiver instance to scene
scene.add(rx)

tx.look_at(rx) # Transmitter points towards receiver

In [ ]:
antenna_pattern

In [ ]:
sionna.rt.antenna.visualize(antenna_pattern)

In [ ]:
scene.frequency = 2.14e9 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = True # If set to False, ray tracing will be done per antenna element (slower for large arrays)

In [ ]:
# Select an example object from the scene
so = scene.get("Altes_Rathaus-itu_marble")

# Print name of assigned radio material for different frequenies
for f in [3.5e9, 2.14e9]: # Print for differrent frequencies
    scene.frequency = f
    print(f"\nRadioMaterial: {so.radio_material.name} @ {scene.frequency/1e9:.2f}GHz")
    print("Conductivity:", so.radio_material.conductivity.numpy())
    print("Relative permittivity:", so.radio_material.relative_permittivity.numpy())
    print("Complex relative permittivity:", so.radio_material.complex_relative_permittivity.numpy())
    print("Relative permeability:", so.radio_material.relative_permeability.numpy())
    print("Scattering coefficient:", so.radio_material.scattering_coefficient.numpy())
    print("XPD coefficient:", so.radio_material.xpd_coefficient.numpy())

In [ ]:
# Compute propagation paths
paths = scene.compute_paths(max_depth=5,
                            num_samples=1e6)  # Number of rays shot into directions defined
                                              # by a Fibonacci sphere , too few rays can
                                              # lead to missing paths

# Visualize paths in the scene
if no_preview:
    scene.render("my_cam", paths=paths, show_devices=True, show_paths=True, resolution=resolution);

In [ ]:
%%skip_if no_preview
scene.preview(paths, show_devices=True, show_paths=True) # Use the mouse to focus on the visualized paths

In [ ]:
# Show the coordinates of the starting points of all rays.
# These coincide with the location of the transmitters.
print("Source coordinates: ", paths.sources.numpy())
print("Transmitter coordinates: ", list(scene.transmitters.values())[0].position.numpy())

# Show the coordinates of the endpoints of all rays.
# These coincide with the location of the receivers.
print("Target coordinates: ",paths.targets.numpy())
print("Receiver coordinates: ",list(scene.receivers.values())[0].position.numpy())

# Show the types of all paths:
# 0 - LoS, 1 - Reflected, 2 - Diffracted, 3 - Scattered
# Note that Diffraction and scattering are turned off by default.
print("Path types: ", paths.types.numpy())

In [ ]:
# We can now access for every path the channel coefficient, the propagation delay,
# as well as the angles of departure and arrival, respectively (zenith and azimuth).

# Let us inspect a specific path in detail
path_idx = 4 # Try out other values in the range [0, 13]

# For a detailed overview of the dimensions of all properties, have a look at the API documentation
print(f"\n--- Detailed results for path {path_idx} ---")
print(f"Channel coefficient: {paths.a[0,0,0,0,0,path_idx, 0].numpy()}")
print(f"Propagation delay: {paths.tau[0,0,0,path_idx].numpy()*1e6:.5f} us")
print(f"Zenith angle of departure: {paths.theta_t[0,0,0,path_idx]:.4f} rad")
print(f"Azimuth angle of departure: {paths.phi_t[0,0,0,path_idx]:.4f} rad")
print(f"Zenith angle of arrival: {paths.theta_r[0,0,0,path_idx]:.4f} rad")
print(f"Azimuth angle of arrival: {paths.phi_r[0,0,0,path_idx]:.4f} rad")

In [ ]:
# Default parameters in the PUSCHConfig
subcarrier_spacing = 15e3
fft_size = 48

In [ ]:
# Print shape of channel coefficients before the application of Doppler shifts
# The last dimension corresponds to the number of time steps which defaults to one
# as there is no mobility
print("Shape of `a` before applying Doppler shifts: ", paths.a.shape)

# Apply Doppler shifts
paths.apply_doppler(sampling_frequency=subcarrier_spacing, # Set to 15e3 Hz
                    num_time_steps=14, # Number of OFDM symbols
                    tx_velocities=[3.,0,0], # We can set additional tx speeds
                    rx_velocities=[0,7.,0]) # Or rx speeds

print("Shape of `a` after applying Doppler shifts: ", paths.a.shape)

a, tau = paths.cir()
print("Shape of tau: ", tau.shape)

In [ ]:
t = tau[0,0,0,:]/1e-9 # Scale to ns
a_abs = np.abs(a)[0,0,0,0,0,:,0]
a_max = np.max(a_abs)
# Add dummy entry at start/end for nicer figure
t = np.concatenate([(0.,), t, (np.max(t)*1.1,)])
a_abs = np.concatenate([(np.nan,), a_abs, (np.nan,)])

# And plot the CIR
plt.figure()
plt.title("Channel impulse response realization")

plt.stem(t, a_abs)
plt.xlim([0, np.max(t)])
plt.ylim([-2e-6, a_max*1.1])
plt.xlabel(r"$\tau$ [ns]")
plt.ylabel(r"$|a|$");
plt.show()

In [ ]:
# Disable normalization of delays
paths.normalize_delays = False

# Get only the LoS path
_, tau = paths.cir(los=True, reflection=False)
print("Delay of first path without normalization: ", np.squeeze(tau))

paths.normalize_delays = True
_, tau = paths.cir(los=True, reflection=False)
print("Delay of first path with normalization: ", np.squeeze(tau))

In [ ]:
# Compute frequencies of subcarriers and center around carrier frequency
frequencies = subcarrier_frequencies(fft_size, subcarrier_spacing)

# Compute the frequency response of the channel at frequencies.
h_freq = cir_to_ofdm_channel(frequencies,
                             a,
                             tau,
                             normalize=True) # Non-normalized includes path-loss

# Verify that the channel power is normalized
h_avg_power = tf.reduce_mean(tf.abs(h_freq)**2).numpy()

print("Shape of h_freq: ", h_freq.shape)
print("Average power h_freq: ", h_avg_power) # Channel is normalized

In [ ]:
# Placeholder for tx signal of shape
# [batch size, num_tx, num_tx_ant, num_ofdm_symbols, fft_size]
x = tf.zeros([h_freq.shape.as_list()[i] for i in [0,3,4,5,6]], tf.complex64)

no = 0.1 # noise variance

# Init channel layer
channel = ApplyOFDMChannel(add_awgn=True)

# Apply channel
y = channel([x, h_freq, no])

# [batch size, num_rx, num_rx_ant, num_ofdm_symbols, fft_size]
print(y.shape)

In [ ]:
# Init pusch_transmitter
pusch_config = PUSCHConfig()

# Instantiate a PUSCHTransmitter from the PUSCHConfig
pusch_transmitter = PUSCHTransmitter(pusch_config)

# Create a PUSCHReceiver using the PUSCHTransmitter
pusch_receiver = PUSCHReceiver(pusch_transmitter)

In [ ]:
# Simulate transmissions over the
batch_size = 100 # h_freq is broadcast, i.e., same CIR for all samples but different AWGN realizations
ebno_db = 2. # SNR in dB

no = ebnodb2no(ebno_db,
               pusch_transmitter._num_bits_per_symbol,
               pusch_transmitter._target_coderate,
               pusch_transmitter.resource_grid)

x, b = pusch_transmitter(batch_size) # Generate transmit signal and info bits

y = channel([x, h_freq, no]) # Simulate channel output

b_hat = pusch_receiver([y, no]) # Recover the info bits

# Compute BER
print(f"BER: {compute_ber(b, b_hat).numpy():.5f}")

In [ ]:
max_depths = 10 # evaluate performance up to 10 reflections
depths = range(1,max_depths+1)
ts = []
pl_avg = []
for d in depths:
    # save start time
    t = time.time()
    # run the ray tracer
    paths = scene.compute_paths(max_depth=d)
    # and measure the required time interval
    ts.append(time.time()-t)

In [ ]:
# and plot results
plt.figure()
plt.plot(depths, ts, color="b");
plt.xlabel("Max. depth")
plt.ylabel("Runtime (s)", color="b")
plt.grid(which="both")
plt.xlim([1, max_depths]);

In [ ]:
t = time.time()
paths = scene.compute_paths(max_depth=3, diffraction=False)
print("Time without diffraction and scattering:" , time.time()-t)

t = time.time()
paths = scene.compute_paths(max_depth=3, diffraction=True)
print("Time with diffraction:" , time.time()-t)

t = time.time()
paths = scene.compute_paths(max_depth=3, scattering=True)
print("Time with scattering:" , time.time()-t)

In [ ]:
# Remove old transmitter and add new one
scene.remove("tx")

tx = Transmitter(name="tx",
                 position=[-210,73,105], # top of Frauenkirche
                 orientation=[0,0,0])
scene.add(tx)

# We could have alternatively modified the properties position and orientation of the existing transmitter
#scene.get("tx").position = [-210,73,105]
#scene.get("tx").orientation = [0,0,0]

In [ ]:
 # Open 3D preview (only works in Jupyter notebook)
if no_preview:
    scene.render(camera="scene-cam-0", num_samples=512, resolution=resolution);

In [ ]:
 %%skip_if no_preview
scene.preview()

## Coverage Map Component

In [ ]:
cm = scene.coverage_map(max_depth=5,
                        diffraction=True, # Disable to see the effects of diffraction
                        cm_cell_size=(5., 5.), # Grid size of coverage map cells in m
                        combining_vec=None,
                        precoding_vec=None,
                        num_samples=int(20e6)) # Reduce if your hardware does not have enough memory

In [ ]:
# Create new camera
tx_pos = scene.transmitters["tx"].position.numpy()
bird_pos = tx_pos.copy()
bird_pos[-1] = 1000 # Set height of coverage map to 1000m above tx
bird_pos[-2]-= 0.01 # Slightly move the camera for correct orientation

# Create new camera
bird_cam = Camera("birds_view", position=bird_pos, look_at=tx_pos)

scene.add(bird_cam)

if no_preview:
    scene.render(camera="birds_view", coverage_map=cm, num_samples=512, resolution=resolution);

In [ ]:
%%skip_if no_preview
scene.preview(coverage_map=cm)

In [ ]:
cm.show(tx=0); # If multiple transmitters exist, tx selects for which transmitter the cm is shown

In [ ]:
# System parameters
subcarrier_spacing = 30e3
num_time_steps = 14 # Total number of ofdm symbols per slot

num_tx = 4 # Number of users
num_rx = 1 # Only one receiver considered
num_tx_ant = 4 # Each user has 4 antennas
num_rx_ant = 16 # The receiver is equipped with 16 antennas

# batch_size for CIR generation
batch_size_cir = 1000

In [ ]:
# Remove old tx from scene
scene.remove("tx")

scene.synthetic_array = True # Emulate multiple antennas to reduce ray tracing complexity

# Transmitter (=basestation) has an antenna pattern from 3GPP 38.901
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=int(num_rx_ant/2), # We want to transmitter to be equiped with the 16 rx antennas
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="cross")

# Create transmitter
tx = Transmitter(name="tx",
                 position=[8.5,21,27],
                 look_at=[45,90,1.5]) # optional, defines view direction
scene.add(tx)

In [ ]:
max_depth = 5 # Defines max number of ray interactions

# Update coverage_map
cm = scene.coverage_map(max_depth=max_depth,
                        diffraction=True,
                        cm_cell_size=(1., 1.),
                        combining_vec=None,
                        precoding_vec=None,
                        num_samples=int(10e6))

In [ ]:
min_gain_db = -130 # in dB; ignore any position with less than -130 dB path gain
max_gain_db = 0 # in dB; ignore strong paths

# sample points in a 5-400m radius around the receiver
min_dist = 5 # in m
max_dist = 400 # in m

#sample batch_size random user positions from coverage map
ue_pos, _ = cm.sample_positions(num_pos=batch_size_cir,
                                metric="path_gain",
                                min_val_db=min_gain_db,
                                max_val_db=max_gain_db,
                                min_dist=min_dist,
                                max_dist=max_dist)
ue_pos = tf.squeeze(ue_pos)

In [ ]:
# Remove old receivers from scene
scene.remove("rx")
for i in range(batch_size_cir):
    scene.remove(f"rx-{i}")

# Configure antenna array for all receivers (=UEs)
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=int(num_tx_ant/2), # Each receiver is equipped with 4 tx antennas (uplink)
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="iso", # UE orientation is random
                             polarization="cross")

# Create batch_size receivers
for i in range(batch_size_cir):
    rx = Receiver(name=f"rx-{i}",
                  position=ue_pos[i], # Random position sampled from coverage map
                  )
    scene.add(rx)

# And visualize the scene
if no_preview:
    scene.render("birds_view", show_devices=True, resolution=resolution);

In [ ]:
%%skip_if no_preview
scene.preview(show_devices=True, coverage_map=cm)

In [ ]:
target_num_cirs = 5000 # Defines how many different CIRS are generated.
# Remark: some path are removed if no path was found for this position

max_depth = 5
min_gain_db = -130 # in dB / ignore any position with less than -130 dB path gain
max_gain_db = 0 # in dB / ignore any position with more than 0 dB path gain

# Sample points within a 10-400m radius around the transmitter
min_dist = 10 # in m
max_dist = 400 # in m

# Placeholder to gather channel impulse reponses
a = None
tau = None

# Each simulation returns batch_size_cir results
num_runs = int(np.ceil(target_num_cirs/batch_size_cir))
for idx in range(num_runs):
    print(f"Progress: {idx+1}/{num_runs}", end="\r")

    # Sample random user positions
    ue_pos, _ = cm.sample_positions(
                        num_pos=batch_size_cir,
                        metric="path_gain",
                        min_val_db=min_gain_db,
                        max_val_db=max_gain_db,
                        min_dist=min_dist,
                        max_dist=max_dist)
    ue_pos = tf.squeeze(ue_pos)

    # Update all receiver positions
    for idx in range(batch_size_cir):
        scene.receivers[f"rx-{idx}"].position = ue_pos[idx]

    # Simulate CIR
    paths = scene.compute_paths(
                    max_depth=max_depth,
                    diffraction=True,
                    num_samples=1e6) # shared between all tx in a scene

    # Transform paths into channel impulse responses
    paths.reverse_direction = True # Convert to uplink direction
    paths.apply_doppler(sampling_frequency=subcarrier_spacing,
                        num_time_steps=14,
                        tx_velocities=[0.,0.,0],
                        rx_velocities=[3.,3.,0])

    # We fix here the maximum number of paths to 75 which ensures
    # that we can simply concatenate different channel impulse reponses
    a_, tau_ = paths.cir(num_paths=75)
    del paths # Free memory

    if a is None:
        a = a_.numpy()
        tau = tau_.numpy()
    else:
        # Concatenate along the num_tx dimension
        a = np.concatenate([a, a_], axis=3)
        tau = np.concatenate([tau, tau_], axis=2)

del cm # Free memory

# Exchange the num_tx and batchsize dimensions
a = np.transpose(a, [3, 1, 2, 0, 4, 5, 6])
tau = np.transpose(tau, [2, 1, 0, 3])

# Remove CIRs that have no active link (i.e., a is all-zero)
p_link = np.sum(np.abs(a)**2, axis=(1,2,3,4,5,6))
a = a[p_link>0.,...]
tau = tau[p_link>0.,...]

print("Shape of a:", a.shape)
print("Shape of tau: ", tau.shape)

In [ ]:
class CIRGenerator:
    """Creates a generator from a given dataset of channel impulse responses.

    The generator samples ``num_tx`` different transmitters from the given path
    coefficients `a` and path delays `tau` and stacks the CIRs into a single tensor.

    Note that the generator internally samples ``num_tx`` random transmitters
    from the dataset. For this, the inputs ``a`` and ``tau`` must be given for
    a single transmitter (i.e., ``num_tx`` =1) which will then be stacked
    internally.

    Parameters
    ----------
    a : [batch size, num_rx, num_rx_ant, 1, num_tx_ant, num_paths, num_time_steps], complex
        Path coefficients per transmitter.

    tau : [batch size, num_rx, 1, num_paths], float
        Path delays [s] per transmitter.

    num_tx : int
        Number of transmitters

    Output
    -------
    a : [batch size, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, num_time_steps], tf.complex
        Path coefficients

    tau : [batch size, num_rx, num_tx, num_paths], tf.float
        Path delays [s]
    """

    def __init__(self,
                 a,
                 tau,
                 num_tx):

        # Copy to tensorflow
        self._a = tf.constant(a, tf.complex64)
        self._tau = tf.constant(tau, tf.float32)
        self._dataset_size = self._a.shape[0]

        self._num_tx = num_tx

    def __call__(self):

        # Generator implements an infinite loop that yields new random samples
        while True:
            # Sample 4 random users and stack them together
            idx,_,_ = tf.random.uniform_candidate_sampler(
                            tf.expand_dims(tf.range(self._dataset_size, dtype=tf.int64), axis=0),
                            num_true=self._dataset_size,
                            num_sampled=self._num_tx,
                            unique=True,
                            range_max=self._dataset_size)

            a = tf.gather(self._a, idx)
            tau = tf.gather(self._tau, idx)

            # Transpose to remove batch dimension
            a = tf.transpose(a, (3,1,2,0,4,5,6))
            tau = tf.transpose(tau, (2,1,0,3))

            # And remove batch-dimension
            a = tf.squeeze(a, axis=0)
            tau = tf.squeeze(tau, axis=0)

            yield a, tau

In [ ]:
batch_size = 20 # Must be the same for the BER simulations as CIRDataset returns fixed batch_size

# Init CIR generator
cir_generator = CIRGenerator(a,
                             tau,
                             num_tx)
# Initialises a channel model that can be directly used by OFDMChannel layer
channel_model = CIRDataset(cir_generator,
                           batch_size,
                           num_rx,
                           num_rx_ant,
                           num_tx,
                           num_tx_ant,
                           75,
                           num_time_steps)

# Delete to free memory
del a, tau

In [ ]:
# We need to enable sionna.config.xla_compat before we can use
# tf.function with jit_compile=True.
# See https://nvlabs.github.io/sionna/api/config.html#sionna.Config.xla_compat
sionna.config.xla_compat=False # not supported in CIRDataset

class Model(tf.keras.Model):
    """Simulate PUSCH transmissions over a 3GPP 38.901 model.

    This model runs BER simulations for a multi-user MIMO uplink channel
    compliant with the 5G NR PUSCH specifications.
    You can pick different scenarios, i.e., channel models, perfect or
    estimated CSI, as well as different MIMO detectors (LMMSE or KBest).

    Parameters
    ----------
    channel_model : :class:`~sionna.channel.ChannelModel` object
        An instance of a :class:`~sionna.channel.ChannelModel` object, such as
        :class:`~sionna.channel.RayleighBlockFading` or
        :class:`~sionna.channel.tr38901.UMi` or
        :class:`~sionna.channel.CIRDataset`.

    perfect_csi : bool
        Determines if perfect CSI is assumed or if the CSI is estimated

    detector : str, one of ["lmmse", "kbest"]
        MIMO detector to be used. Note that each detector has additional
        parameters that can be configured in the source code of the _init_ call.

    Input
    -----
    batch_size : int
        Number of simultaneously simulated slots

    ebno_db : float
        Signal-to-noise-ratio

    Output
    ------
    b : [batch_size, num_tx, tb_size], tf.float
        Transmitted information bits

    b_hat : [batch_size, num_tx, tb_size], tf.float
        Decoded information bits
    """
    def __init__(self,
                 channel_model,
                 perfect_csi, # bool
                 detector,    # "lmmse", "kbest"
                ):
        super().__init__()

        self._channel_model = channel_model
        self._perfect_csi = perfect_csi

        # System configuration
        self._num_prb = 16
        self._mcs_index = 14
        self._num_layers = 1
        self._mcs_table = 1
        self._domain = "freq"

        # Below parameters must equal the Path2CIR parameters
        self._num_tx_ant = 4
        self._num_tx = 4
        self._subcarrier_spacing = 30e3 # must be the same as used for Path2CIR

        # PUSCHConfig for the first transmitter
        pusch_config = PUSCHConfig()
        pusch_config.carrier.subcarrier_spacing = self._subcarrier_spacing/1000
        pusch_config.carrier.n_size_grid = self._num_prb
        pusch_config.num_antenna_ports = self._num_tx_ant
        pusch_config.num_layers = self._num_layers
        pusch_config.precoding = "codebook"
        pusch_config.tpmi = 1
        pusch_config.dmrs.dmrs_port_set = list(range(self._num_layers))
        pusch_config.dmrs.config_type = 1
        pusch_config.dmrs.length = 1
        pusch_config.dmrs.additional_position = 1
        pusch_config.dmrs.num_cdm_groups_without_data = 2
        pusch_config.tb.mcs_index = self._mcs_index
        pusch_config.tb.mcs_table = self._mcs_table

        # Create PUSCHConfigs for the other transmitters by cloning of the first PUSCHConfig
        # and modifying the used DMRS ports.
        pusch_configs = [pusch_config]
        for i in range(1, self._num_tx):
            pc = pusch_config.clone()
            pc.dmrs.dmrs_port_set = list(range(i*self._num_layers, (i+1)*self._num_layers))
            pusch_configs.append(pc)

        # Create PUSCHTransmitter
        self._pusch_transmitter = PUSCHTransmitter(pusch_configs, output_domain=self._domain)

        # Create PUSCHReceiver
        rx_tx_association = np.ones([1, self._num_tx], bool)
        stream_management = StreamManagement(rx_tx_association,
                                             self._num_layers)

        assert detector in["lmmse", "kbest"], "Unsupported MIMO detector"
        if detector=="lmmse":
            detector = LinearDetector(equalizer="lmmse",
                                      output="bit",
                                      demapping_method="maxlog",
                                      resource_grid=self._pusch_transmitter.resource_grid,
                                      stream_management=stream_management,
                                      constellation_type="qam",
                                      num_bits_per_symbol=pusch_config.tb.num_bits_per_symbol)
        elif detector=="kbest":
            detector = KBestDetector(output="bit",
                                     num_streams=self._num_tx*self._num_layers,
                                     k=64,
                                     resource_grid=self._pusch_transmitter.resource_grid,
                                     stream_management=stream_management,
                                     constellation_type="qam",
                                     num_bits_per_symbol=pusch_config.tb.num_bits_per_symbol)

        if self._perfect_csi:
            self._pusch_receiver = PUSCHReceiver(self._pusch_transmitter,
                                                 mimo_detector=detector,
                                                 input_domain=self._domain,
                                                 channel_estimator="perfect")
        else:
            self._pusch_receiver = PUSCHReceiver(self._pusch_transmitter,
                                                 mimo_detector=detector,
                                                 input_domain=self._domain)


        # Configure the actual channel
        self._channel = OFDMChannel(
                            self._channel_model,
                            self._pusch_transmitter.resource_grid,
                            normalize_channel=True,
                            return_channel=True)

    # XLA currently not supported by the CIRDataset function
    @tf.function(jit_compile=False)
    def call(self, batch_size, ebno_db):

        x, b = self._pusch_transmitter(batch_size)
        no = ebnodb2no(ebno_db,
                       self._pusch_transmitter._num_bits_per_symbol,
                       pusch_transmitter._target_coderate,
                       pusch_transmitter.resource_grid)
        y, h = self._channel([x, no])
        if self._perfect_csi:
            b_hat = self._pusch_receiver([y, h, no])
        else:
            b_hat = self._pusch_receiver([y, no])
        return b, b_hat

In [ ]:
ebno_db = 10.
e2e_model = Model(channel_model,
                  perfect_csi=False, # bool
                  detector="lmmse")  # "lmmse", "kbest"

# We can draw samples from the end-2-end link-level simulations
b, b_hat = e2e_model(batch_size, ebno_db)

In [ ]:
ebno_db = np.arange(-3, 18, 2) # sim SNR range
ber_plot = PlotBER(f"Site-Specific MU-MIMO 5G NR PUSCH")

for detector in ["lmmse", "kbest"]:
    for perf_csi in [True, False]:
        e2e_model = Model(channel_model,
                          perfect_csi=perf_csi,
                          detector=detector)
        # define legend
        csi = "Perf. CSI" if perf_csi else "Imperf. CSI"
        det = "K-Best" if detector=="kbest" else "LMMSE"
        l = det + " " + csi
        ber_plot.simulate(
                    e2e_model,
                    ebno_dbs=ebno_db, # SNR to simulate
                    legend=l, # legend string for plotting
                    max_mc_iter=500,
                    num_target_block_errors=2000,
                    batch_size=batch_size, # batch-size per Monte Carlo run
                    soft_estimates=False, # the model returns hard-estimates
                    early_stop=True,
                    show_fig=False,
                    add_bler=True,
                    forward_keyboard_interrupt=True);

In [ ]:
# and show figure
ber_plot(show_bler=True, show_ber=False, ylim=[1e-4,1], xlim=[-3,17])